In [1]:
import pandas as pd

In [2]:

# Load the agent's saved findings from notebook 03


with open("/Users/akashkumarsamantray/BI_Agent/bi-insight-agent/agent_findings.txt", "r") as f:
    agent_findings = f.read()

print(agent_findings)

# Statistical Test Results Analysis: Test vs Control

## Key Finding: Trade-offs Accompanying Completion Rate Improvement

**The Test group shows higher completion rates across all segments, BUT this improvement comes with measurable costs in both duration and backward navigation.**

### Overall Pattern Across All Segments

| Metric | Test vs Control | Magnitude |
|--------|----------------|-----------|
| **Completion Rate** | Higher in Test | +3.2 to +4.1 percentage points |
| **Process Duration** | Higher in Test | +24 to +47 seconds longer |
| **Backward Navigation Rate** | Higher in Test | +4.2 to +9.5 percentage points |

All differences are statistically significant (p < 0.00001).

---

## Detailed Findings by Metric

### 1. Completion Rate
Test group shows consistently higher completion rates:
- **High age**: 67.1% (Test) vs 63.8% (Control) — **+3.2 pts**
- **Low age**: 71.5% (Test) vs 67.4% (Control) — **+4.1 pts**
- **High tenure**: 68.5% (Test) vs 65.0% (Control) — **+3.5 pts

In [3]:

# Write out the manual baseline numbers from notebook 02

# These are the numbers we calculated by hand in notebook 02.
# We're typing them here directly since they're the fixed "ground truth"
# we're checking the agent against.

manual_findings = {
    "completion_rate": {"Control": 0.6558, "Test": 0.6929},
    "duration_sec": {"Control": 293.76, "Test": 328.36},
    "backward_navigation_rate": {"Control": 0.2609, "Test": 0.3341}
}

for metric, values in manual_findings.items():
    print(f"{metric}: Control = {values['Control']}, Test = {values['Test']}")

completion_rate: Control = 0.6558, Test = 0.6929
duration_sec: Control = 293.76, Test = 328.36
backward_navigation_rate: Control = 0.2609, Test = 0.3341


In [4]:

# CELL 4 — Build a simple side-by-side comparison table

# This just lays out manual vs agent findings in one place so it's easy to read
# and easy to put in a report or README later.

comparison_table = pd.DataFrame({
    "Metric": ["Completion Rate", "Duration (sec)", "Backward Navigation Rate"],
    "Manual - Control": [0.6558, 293.76, 0.2609],
    "Manual - Test": [0.6929, 328.36, 0.3341],
    "Direction Found by Agent": ["Test higher", "Test higher (longer)", "Test higher (more backtracking)"],
    "Matches Manual?": ["Yes", "Yes", "Yes"]
})

comparison_table

,Metric,Manual - Control,Manual - Test,Direction Found by Agent,Matches Manual?
0,Completion Rate,0.6558,0.6929,Test higher,Yes
1,Duration (sec),293.7600,328.3600,Test higher (longer),Yes
2,Backward Navigation Rate,0.2609,0.3341,Test higher (more backtracking),Yes


In [5]:

# CELL 5 — Check: did the agent correctly identify the "hidden cost" story?

# This isn't a numeric check — it's a manual read of the agent's text output,
# checking whether it mentioned that the completion-rate win came with a cost.
# We're looking for key phrases that show it connected the dots, not just listed numbers.

keywords_to_check = ["duration", "backward", "cost", "longer", "trade-off", "however", "but"]

found_keywords = [word for word in keywords_to_check if word.lower() in agent_findings.lower()]

print("Keywords found in agent's explanation:", found_keywords)

if "duration" in found_keywords or "backward" in found_keywords:
    print("\n✅ Agent successfully connected completion rate to the friction cost.")
else:
    print("\n⚠️ Agent may have missed the friction/cost connection — review its output manually.")

Keywords found in agent's explanation: ['duration', 'backward', 'cost', 'longer', 'trade-off', 'however', 'but']

✅ Agent successfully connected completion rate to the friction cost.


In [6]:

# CELL 6 — Check: did the agent respect the guardrails we gave it?


# Rule 1 was: don't trust unreliable segments (100%/0% with missing data)
# Rule 2 was: don't recommend targeting specific demographic groups
# Rule 4 was: describe WHAT happened, not WHY (no guessing at motivations)

guardrail_violations = []

if "target" in agent_findings.lower() and "recommend" in agent_findings.lower():
    guardrail_violations.append("Possible targeting recommendation found — review manually")

motivation_words = ["comfortable", "confusion", "struggling", "prefer", "frustrat"]
for word in motivation_words:
    if word in agent_findings.lower():
        guardrail_violations.append(f"Possible motivation-guessing language found: '{word}'")

if guardrail_violations:
    print("⚠️ Guardrail check found possible issues:")
    for v in guardrail_violations:
        print(" -", v)
else:
    print("✅ No guardrail violations detected in this run.")

✅ No guardrail violations detected in this run.


In [7]:

# CELL 7 — Save the final comparison table for the README / portfolio

comparison_table.to_csv("validation_comparison.csv", index=False)
print("Saved validation_comparison.csv")

Saved validation_comparison.csv


In [ ]:
## Summary of findings — Validation & Comparison

**Objective:** Check whether the automated agent (notebook 03) independently reached the
same conclusions as the manual analysis (notebook 02).

**Result:** The agent correctly identified, without being told the answer in advance:
- Test group has a higher completion rate than Control
- Test group takes longer on average (duration)
- Test group shows more backward navigation (friction)
- These three findings together tell the same "hidden cost" story found manually:
  the completion-rate improvement is real, but comes with a measurable friction cost.

**Guardrails validated:**
- Sample-size check correctly prevented two unreliable segments (`gendr` X group,
  `num_accts` Low group) from being reported as real findings
- Prompt rules successfully prevented demographic-targeting recommendations
- Prompt rules mostly prevented motivation-guessing language, after one round of refinement

**Conclusion:** This validates the core premise of the project — an agent, when properly
guardrailed against small-sample noise and steered away from speculative interpretation,
can automate root-cause investigative work that would otherwise take an analyst hours to
do manually, and can do so reliably enough to match a verified human baseline.